# AELIONIX BLACKFORGE — Phase 14 Colab Validation

This notebook performs a deterministic, one-click validation of the **Attack
Graph & Autonomous Planner Foundation** (Phase 14).

The attack-graph layer is an **evidence-driven, bounded, descriptive reasoning
layer** — it is explicitly *not* an attack engine. It derives a mission-scoped
analytical graph from the world model using deterministic rules
(`ag_exposure`, `ag_dependency`, `ag_reachability`, `ag_access`,
`ag_constraint`, `ag_correlation_discrepancy`, `ag_hypothesis`) and then
produces *advisory assessment plans* that only reference **registered, low-risk
BLACKFORGE capabilities**.

The notebook validates, over standalone fixture world models:

* **derived, not invented** — every node is a view of a canonical world
  entity (`ag_entity_mirror` provenance); every edge carries the exact
  world-model relationships and evidence it was derived from
* **weaker-than-source guarantees** — derived edges stay `INFERRED`, hypotheses
  are `HYPOTHESIZED`/LOW with an explicit "potential relationship only; not
  exploitation" assumption, and no graph record can reach `VALIDATED` without
  validated sources
* **no offensive semantics** — the graph vocabulary (`exposes`, `requires`,
  `depends_on`, `reachable_from`, `provides_access_to`, `constrains`,
  `blocked_by`, `potentially_leads_to`, `requires_validation`) never contains
  exploitation, compromise, escalation, or bypass predicates
* **evidence-gap reasoning** — `compute_gaps`, `missing_prerequisites`, and
  deterministic path classification (`INFERRED_PATH` / `HYPOTHESIZED_PATH` /
  `BLOCKED_PATH` / …) that never upgrades a derived path
* **explainable scoring** — `assess` returns a deterministic, explainable
  0..1 priority score (LOW / MODERATE / ELEVATED / HIGH) for ordering
  authorized evidence-gathering work — never a probability and never an
  exploitability claim
* **fail-closed planner** — `NO_ACTION_AVAILABLE`, `NO_GAPS`,
  `INSUFFICIENT_SCOPE`, and `PLAN_CAPPED` are first-class outcomes; the planner
  only *selects* registered capabilities and never executes anything
* **bounded budgets** — `max_steps`, `max_candidates`, `max_graph_depth`,
  `max_replans`, and an idle-IDE budget hard-cap per plan
* **SQLite persistence** — graphs (with supersede history) and full plan
  payloads survive a process restart through a fresh repository connection

> Run all cells top-to-bottom. No GPU, no external services, no credentials.
> The notebook fails loudly on any check.


---

In [ ]:
import sys
import platform

print("Blackforge Phase 14 Colab Validation (Attack Graph & Autonomous Planner Foundation)")
print("=" * 60)
print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("Platform:", platform.platform())
print("Architecture:", platform.machine())
print("=" * 60)

assert sys.version_info >= (3, 10), f"Blackforge requires Python 3.10+, got {sys.version}"
print("Python version check: PASS")

---

In [ ]:
from pathlib import Path
import subprocess
import sys
import os
import shutil

# -- Configuration (edit here if fork changes) ---------------------------
REPO_URL = "https://github.com/Sagelord00000001/Blackforge.git"
REPO_DIR = Path("/content/blackforge")
# -----------------------------------------------------------------------

if REPO_DIR.exists() and (REPO_DIR / "blackforge" / "__init__.py").exists():
    print(f"Repository already exists at {REPO_DIR}, updating...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True,
    )

os.chdir(str(REPO_DIR))
print(f"Repository ready at {REPO_DIR}")

---

In [ ]:
import subprocess

try:
    commit = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    print("Commit:", commit)
except Exception as e:
    print("Commit unavailable (expected in scratch checkouts):", e)

---

In [ ]:
!pip install hatchling --quiet
!pip install -e ".[dev]" --quiet

---

In [ ]:
import importlib

modules = [
    "blackforge",
    "blackforge.core.config",
    "blackforge.core.errors",
    "blackforge.core.types",
    "blackforge.runtime.bootstrap",
    "blackforge.capabilities.registry",
    "blackforge.capabilities.models",
    "blackforge.capabilities.interface",
    "blackforge.authorization",
    "blackforge.scope.models",
    "blackforge.evidence",
    "blackforge.evidence.models",
    "blackforge.evidence.store",
    "blackforge.evidence.repository",
    "blackforge.world_model",
    "blackforge.world_model.models",
    "blackforge.world_model.query",
    "blackforge.world_model.repository",
    "blackforge.world_model.store",
    "blackforge.attack_graph",
    "blackforge.attack_graph.models",
    "blackforge.attack_graph.rules",
    "blackforge.attack_graph.builder",
    "blackforge.attack_graph.graph",
    "blackforge.attack_graph.query",
    "blackforge.attack_graph.actions",
    "blackforge.attack_graph.scoring",
    "blackforge.attack_graph.planner",
    "blackforge.attack_graph.repository",
    "blackforge.attack_graph.materializer",
    "blackforge.attack_graph.validation",
]

_import_failures = []
for module in modules:
    try:
        importlib.import_module(module)
    except Exception as e:
        _import_failures.append((module, str(e)))

if _import_failures:
    for mod, err in _import_failures:
        print(f"  FAIL: {mod} -- {err}")
    raise RuntimeError(f"Import health check failed: {len(_import_failures)} module(s)")

print(f"Blackforge imports OK ({len(modules)} modules verified).")
print("Attack graph & planner module imports: PASS")

---

In [ ]:
import subprocess
import sys
import os

print("Running automated test suite...")
# The LLM/torch-heavy files are excluded: importing the HF provider pulls
# ~2GB of torch memory and can SIGKILL the kernel on CPU runtimes. Those
# tests are validated locally and in the Phase 1 notebook.
# The subprocess runs against pristine defaults: BLACKFORGE_* env overrides
# from the runtime are stripped so the suite behaves exactly like CI and no
# ambient config skews assertions (e.g. log level / DB path).
_test_env = {k: v for k, v in os.environ.items() if not k.startswith("BLACKFORGE_")}
result = subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q", "--tb=short",
        "--ignore=tests/test_huggingface_provider.py",
        "--ignore=tests/test_loader.py",
        "--ignore=tests/test_smoke_real_model.py",
    ],
    capture_output=True, text=True, cwd=str(REPO_DIR), env=_test_env,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:] if result.stderr else "")
    raise RuntimeError(f"pytest failed with exit code {result.returncode}")

print("Automated test suite: PASS")

---

In [ ]:
import os
from pathlib import Path

DBROOT = Path("data/phase14_colab").resolve()
DBROOT.mkdir(parents=True, exist_ok=True)
os.environ["BLACKFORGE_DB_PATH"] = str(DBROOT / "blackforge.db")
os.environ["BLACKFORGE_MEMORY_DB_PATH"] = str(DBROOT / "memory.db")
os.environ["BLACKFORGE_EVIDENCE_DB_PATH"] = str(DBROOT / "evidence.db")
os.environ["BLACKFORGE_WORLD_MODEL_DB_PATH"] = str(DBROOT / "world_model.db")
for _p in (DBROOT / "evidence.db", DBROOT / "world_model.db"):
    _p.unlink(missing_ok=True)

from blackforge.runtime.bootstrap import bootstrap

app = bootstrap()
assert app.healthy(), "Blackforge health check failed"
verification = app.verify()
for key in ("config_loaded", "mission_manager_ready", "capability_registry_ready",
            "memory_ready", "evidence_store_ready", "evidence_memory_link_ready",
            "world_model_ready", "recon_ready", "webapi_ready", "auth_ready",
            "business_logic_ready", "network_ready", "identity_ready",
            "authorization_ready", "model_router_ready", "cloud_ready",
            "container_ready"):
    assert verification[key], f"{key} must be True"
assert verification["attack_graph_ready"] is True, "attack_graph_ready must be True"
assert verification["planner_ready"] is True, "planner_ready must be True"
assert len(verification) == 22, len(verification)
assert len(app.capability_registry.list_capabilities()) == 105

BOOTSTRAP_OK = app.healthy() and bool(verification["attack_graph_ready"]) and bool(
    verification["planner_ready"]
)

for k, v in verification.items():
    symbol = "PASS" if v else "FAIL"
    print(f"  [{symbol}] {k}")

print("\nBlackforge bootstrap (attack_graph_ready + planner_ready, 105 capabilities): PASS")

---

In [ ]:
from blackforge.evidence.models import Evidence, EvidenceStatus, EvidenceType, Provenance
from blackforge.evidence.repository import InMemoryEvidenceRepository
from blackforge.evidence.store import EvidenceStore
from blackforge.world_model.models import (
    EntitySpec,
    EntityType,
    EvidenceLinkRef,
    RelationshipSpec,
    RelationshipType,
)
from blackforge.world_model.repository import InMemoryWorldRepository
from blackforge.world_model.store import WorldModelStore
from blackforge.core.types import Confidence
from blackforge.attack_graph import (
    AttackGraphBuilder,
    AttackGraphBuildRequest,
)
from blackforge.attack_graph.models import GraphRelationship, GraphState, PathState
from blackforge.attack_graph.query import AttackGraphQuery, classify_path

MID = "mission_phase14_ag"
SID = "session_phase14_ag"
TARGET = "https://api.aelionix.io/"

# -- Deterministic correlation fixture: an exposed app with a declared-vs-runtime
#    discrepancy and two serving fronts. --------------------------------------
world = WorldModelStore(repository=InMemoryWorldRepository())
evstore = EvidenceStore(repository=InMemoryEvidenceRepository())
ev = evstore.add(
    Evidence(
        mission_id=MID,
        source_capability="http_probe",
        target="api.aelionix.io",
        evidence_type=EvidenceType.OBSERVATION,
        status=EvidenceStatus.OBSERVED,
        confidence=Confidence.HIGH,
        provenance=Provenance(capability_id="http_probe"),
    )
)
main = world.add_entity(
    EntitySpec(mission_id=MID, entity_type=EntityType.APPLICATION, name="api-svc",
               epistemic_status=EvidenceStatus.OBSERVED, confidence=Confidence.HIGH,
               evidence=[EvidenceLinkRef(evidence_id=ev.id)])
).entity
app_ent = world.add_entity(
    EntitySpec(mission_id=MID, entity_type=EntityType.APPLICATION, name="payments-app",
               properties={"url": TARGET}, epistemic_status=EvidenceStatus.OBSERVED,
               confidence=Confidence.HIGH, evidence=[EvidenceLinkRef(evidence_id=ev.id)])
).entity
front = world.add_entity(
    EntitySpec(mission_id=MID, entity_type=EntityType.EDGE_ENDPOINT, name="edge-fe",
               epistemic_status=EvidenceStatus.OBSERVED, confidence=Confidence.HIGH,
               evidence=[EvidenceLinkRef(evidence_id=ev.id)])
).entity
source = world.add_entity(
    EntitySpec(mission_id=MID, entity_type=EntityType.SOURCE_COMPONENT, name="api-svc:source",
               epistemic_status=EvidenceStatus.OBSERVED, confidence=Confidence.HIGH,
               evidence=[EvidenceLinkRef(evidence_id=ev.id)])
).entity
runtime = world.add_entity(
    EntitySpec(mission_id=MID, entity_type=EntityType.SOURCE_COMPONENT, name="api-svc:runtime",
               epistemic_status=EvidenceStatus.OBSERVED, confidence=Confidence.HIGH,
               evidence=[EvidenceLinkRef(evidence_id=ev.id)])
).entity
world.add_relationship(RelationshipSpec(
    mission_id=MID, relationship_type=RelationshipType.DIFFERS_FROM,
    source_entity_id=source.id, target_entity_id=runtime.id, confidence=Confidence.HIGH))
world.add_relationship(RelationshipSpec(
    mission_id=MID, relationship_type=RelationshipType.SERVES,
    source_entity_id=front.id, target_entity_id=app_ent.id, confidence=Confidence.HIGH))
world.add_relationship(RelationshipSpec(
    mission_id=MID, relationship_type=RelationshipType.SERVES,
    source_entity_id=main.id, target_entity_id=app_ent.id, confidence=Confidence.HIGH))

result = AttackGraphBuilder(world_model=world).build(
    AttackGraphBuildRequest(mission_id=MID, session_id=SID))
graph = result.graph

# -- Structural bookkeeping ----------------------------------------------------
assert graph.node_count == 5, graph.node_count
assert graph.edge_count == 5, graph.edge_count
assert result.derived_edge_count == 3, result.derived_edge_count
assert result.hypothesis_edge_count == 2, result.hypothesis_edge_count
assert [r.rule_id for r in result.rules] == [
    "ag_exposure", "ag_dependency", "ag_reachability", "ag_access",
    "ag_constraint", "ag_correlation_discrepancy"]

# -- Provenance: every node is a canonical-world view; edges cite their source
for node in graph.nodes():
    assert node.derivation.rule_id == "ag_entity_mirror", node.derivation.rule_id
    assert node.derivation.source_entity_ids == [str(node.entity_id)]
print("Node provenance (ag_entity_mirror -> canonical world entity): PASS")

rel_by_kind = {e.relationship: e for e in graph.edges()}
assert GraphRelationship.EXPOSES in rel_by_kind
exposure = rel_by_kind[GraphRelationship.EXPOSES]
assert exposure.state == GraphState.INFERRED
assert exposure.derivation.rule_id == "serves"
assert exposure.derivation.source_relationship_ids, exposure.derivation
print("Exposure edge provenance (derived from world SERVES, INFERRED): PASS")

valid = rel_by_kind[GraphRelationship.REQUIRES_VALIDATION]
assert valid.derivation.rule_id == "ag_correlation_discrepancy"
prereq = valid.prerequisites[0]
assert prereq.description == "declared-vs-runtime discrepancy is validated"
assert prereq.state.value == "unknown"
assert prereq.required_evidence == ["validation_result"]
print("Correlation discrepancy -> REQUIRES_VALIDATION with declared-vs-runtime prereq: PASS")

hyps = [e for e in graph.edges() if e.relationship == GraphRelationship.POTENTIALLY_LEADS_TO]
assert len(hyps) == 2
for h in hyps:
    assert h.state == GraphState.HYPOTHESIZED
    assert h.confidence == Confidence.LOW
    assert h.derivation.rule_id == "ag_hypothesis"
    assert "not exploitation" in " ".join(h.assumptions)
    assert any("must validate" in a for a in h.assumptions)
print("Hypothesis edges: HYPOTHESIZED/LOW, 'potential relationship only; not exploitation': PASS")

# -- Epistemic safety ----------------------------------------------------------
assert all(e.state != GraphState.VALIDATED for e in graph.edges())
assert all(n.state != GraphState.VALIDATED for n in graph.nodes())
from blackforge.attack_graph.validation import graph_contains_offensive_semantics as _gcos
assert _gcos([e.relationship.value for e in graph.edges()]) is False
print("No VALIDATED records and no offensive graph semantics (derived view): PASS")

# -- Path classification never upgrades ----------------------------------------
q = AttackGraphQuery(graph)
open_paths = q.open_paths()
assert len(open_paths) == 2
classifications = {classify_path(graph, p) for p in open_paths}
assert classifications <= {PathState.HYPOTHESIZED_PATH, PathState.INFERRED_PATH}
assert PathState.VALIDATED_PATH not in classifications  # never upgrades
from blackforge.attack_graph.models import GraphPath as _GraphPath
hyp_edge = next(e for e in graph.edges()
                if e.relationship == GraphRelationship.POTENTIALLY_LEADS_TO)
hyp_path = _GraphPath(
    mission_id=MID,
    graph_id=graph.graph_id,
    start_entity_id=hyp_edge.source_entity_id,
    end_entity_id=hyp_edge.target_entity_id,
    edge_ids=[hyp_edge.id],
    length=1,
    state=PathState.HYPOTHESIZED_PATH,
)
assert classify_path(graph, hyp_path) == PathState.HYPOTHESIZED_PATH
print("Path classification: hypothesis stays HYPOTHESIZED_PATH, never upgraded: PASS")

# -- Mission isolation --------------------------------------------------------
other = AttackGraphBuilder(world_model=world).build(
    AttackGraphBuildRequest(mission_id="mission_phase14_other"))
assert other.graph.node_count == 0 and other.graph.edge_count == 0
print("Mission isolation: empty graph for an unrelated mission: PASS")

from blackforge.attack_graph.rules import hypothesis_edges
assert hypothesis_edges(graph.nodes(), graph.edges())  # public post-pass API
DERIVATION_OK = True
print("Graph derivation & evidence integrity: PASS")

---

In [ ]:
from blackforge.capabilities.interface import Capability, CapabilityResult
from blackforge.capabilities.models import CapabilityMeta
from blackforge.capabilities.registry import CapabilityRegistry
from blackforge.core.types import RiskLevel, TargetType
from blackforge.scope.models import Target, TargetScope
from blackforge.attack_graph import AssessmentPlanner, PlanningRequest
from blackforge.attack_graph.models import PlanStatus

class ProbeCapability(Capability):
    def __init__(self, name: str) -> None:
        self._meta = CapabilityMeta(name=name, evidence_types_produced=["validation_result"])

    def meta(self) -> CapabilityMeta:
        return self._meta

    def execute(self, target: str, params: dict | None = None) -> CapabilityResult:
        return CapabilityResult(success=True, output={"ok": True})


def _scope(caps: list[str] | None = None, targets: list[Target] | None = None) -> TargetScope:
    return TargetScope(
        mission_id=MID,
        allowed_targets=targets or [Target(value=TARGET, target_type=TargetType.URL)],
        allowed_capabilities=caps or [],
        max_risk_level=RiskLevel.HIGH,
    )


def _request(graph, *, caps: list[str] | None = None, **kwargs) -> PlanningRequest:
    kwargs.setdefault("session_id", SID)
    return PlanningRequest(mission_id=MID, scope=_scope(caps), graph=graph, **kwargs)


# -- 1) Fail closed with the REAL registry: no validation-result capability is
#        registered, so the planner has nothing to do. NO_ACTION_AVAILABLE is
#        a legitimate, first-class outcome. -----------------------------------
real = AssessmentPlanner(registry=app.capability_registry)
plan_noop = real.plan(_request(graph))
assert plan_noop.status == PlanStatus.NO_ACTION_AVAILABLE, plan_noop.status
assert plan_noop.steps == []
assert plan_noop.graph_id == graph.graph_id
print("Fail-closed default registry -> NO_ACTION_AVAILABLE (no steps): PASS")

# -- 2) Authorized probe registry -> PLAN_READY with exactly the four
#        registered coverage capabilities at the authorized target. ----------
probe = CapabilityRegistry()
for name in ("custom_greeting", "backend_probe", "http_probe", "app_probe"):
    probe.register(ProbeCapability(name))

planner = AssessmentPlanner(registry=probe)
plan = planner.plan(_request(graph))
assert plan.status == PlanStatus.PLAN_READY, plan.status
assert sorted(plan.capability_names) == [
    "app_probe", "backend_probe", "custom_greeting", "http_probe"]
for step in plan.steps:
    assert step.action.target == TARGET
    assert step.action.expected_evidence, step.action.capability_name
    assert step.action.expected_evidence[0].evidence_type in {
        "validation_result", "auth_missing_or_weak",
        "differs_from", "exposure_unconfirmed"}
    assert step.action.rationale, step.action.capability_name
assert plan.max_steps == 10 and plan.max_candidates == 5 and plan.max_graph_depth == 3
print("Authorized probe registry -> PLAN_READY, 4 in-scope registered steps: PASS")

# -- 3) Budget saturation -> PLAN_CAPPED (bounded idle-IDE budget). ----------
plan_capped = planner.plan(_request(graph, ide_budget_hours=0.5))
assert plan_capped.status == PlanStatus.PLAN_CAPPED, plan_capped.status
assert any("saturated" in w or "saturation" in w for w in plan_capped.warnings)
print("Idle-IDE budget saturation -> PLAN_CAPPED: PASS")

# -- 4) max_steps hard-cap ----------------------------------------------------
plan_limited = planner.plan(_request(graph, max_steps=1))
assert len(plan_limited.steps) == 1, len(plan_limited.steps)
assert any("capped" in w for w in plan_limited.warnings)
print("max_steps=1 caps the plan to a single step: PASS")

# -- 5) Out-of-scope target -> INSUFFICIENT_SCOPE (fails closed before work) --
not_in_scope = TargetScope(
    mission_id=MID,
    allowed_targets=[Target(value="https://private.internal/", target_type=TargetType.URL)],
    allowed_capabilities=[],
    max_risk_level=RiskLevel.HIGH,
)
plan_denied = planner.plan(PlanningRequest(
    mission_id=MID, session_id=SID, scope=not_in_scope, graph=graph))
assert plan_denied.status == PlanStatus.INSUFFICIENT_SCOPE, plan_denied.status
assert plan_denied.steps == []
assert any("outside the authorized scope" in w for w in plan_denied.warnings)
print("Out-of-scope target -> INSUFFICIENT_SCOPE (no actions planned): PASS")

# -- 6) Capability gating: allowlist 'none' excludes every probe -> NO_ACTION
plan_gated = planner.plan(_request(graph, caps=["none"]))
assert plan_gated.status == PlanStatus.NO_ACTION_AVAILABLE, plan_gated.status
assert plan_gated.steps == []
assert any("not authorized" in w for w in plan_gated.warnings)
print("Capability allowlist ['none'] -> NO_ACTION_AVAILABLE (nothing authorized): PASS")

# -- 7) Replanning: verified plans get a new id and annotations ---------------
renewed = planner.replan(_request(graph), plan, newly_answered_kinds=["validation_result"])
assert renewed.id != plan.id
assert renewed.note.startswith("replan 1 of 3")
assert any("answered evidence" in w for w in renewed.warnings)
print("Replan produces a new annotated plan (replan 1 of 3): PASS")

PLANNER_OK = (
    plan_noop.status == PlanStatus.NO_ACTION_AVAILABLE
    and plan.status == PlanStatus.PLAN_READY
    and plan_capped.status == PlanStatus.PLAN_CAPPED
    and plan_denied.status == PlanStatus.INSUFFICIENT_SCOPE
)
print("Planner fail-closed + authorized + bounded: PASS")

---

In [ ]:
from blackforge.attack_graph.actions import compute_gaps, expected_evidence_kind, saturation_reached
from blackforge.attack_graph.models import GraphState, PathState
from blackforge.attack_graph.query import AttackGraphQuery, classify_path, missing_prerequisites
from blackforge.attack_graph.scoring import assess
from blackforge.attack_graph.validation import validate_plan
from blackforge.core.errors import PlanningError, GraphValidationError
from pydantic import ValidationError

q = AttackGraphQuery(graph)
path = hyp_path  # deterministic single-edge hypothesis path built above

# -- One evidence gap per derived edge, typed deterministically ---------------
gaps = compute_gaps(graph, [path])
assert len(gaps) >= 1
kinds = {g.evidence_kind for g in gaps}
assert "auth_missing_or_weak" in kinds
assert kinds <= {"validation_result", "auth_missing_or_weak", "differs_from",
                 "exposure_unconfirmed"}, kinds
assert all(g.path_id == str(path.id) for g in gaps)
hyp_edge = next(e for e in graph.edges()
                if e.relationship.value == "potentially_leads_to")
assert expected_evidence_kind(hyp_edge) == "auth_missing_or_weak"
print(f"compute_gaps: {len(gaps)} evidence gaps typed per derived edge: PASS")

# -- Missing prerequisites, deduplicated -------------------------------------
missing = missing_prerequisites(graph, path)
descs = {m.description for m in missing}
assert "discrepancy on payments-app evaluated" in descs
assert any("validated authentication" in d for d in descs)
assert missing_prerequisites(graph, path) == missing_prerequisites(graph, path)
print("missing_prerequisites surfaces validated-auth + discrepancy prereqs, deduped: PASS")

# -- Path classification -----------------------------------------------------
outcome = q.evaluate(path)
assert outcome.state == PathState.HYPOTHESIZED_PATH
assert outcome.length == path.length
assert outcome.missing == [m.description for m in missing]
print("PathOutcome: HYPOTHESIZED_PATH with missing prereqs surfaced: PASS")

# -- Explainable scoring (deterministic; never a probability) ----------------
scale = [assess(1, False, False, 0.3, False, False, 0, 0.3),
         assess(2, False, False, 0.4, False, False, 0, 0.5),
         assess(2, False, False, 0.8, True, False, 1, 0.5),
         assess(3, False, False, 1.0, True, True, 2, 0.95)]
levels = [s.level for s in scale]
assert levels == ["LOW", "MODERATE", "ELEVATED", "HIGH"], levels
for s in scale:
    assert 0.0 <= s.value <= 1.0
    assert s.explain(), "each score must be explainable"
assert assess(3, False, False, 1.0, True, True, 2, 0.95).value ==     assess(3, False, False, 1.0, True, True, 2, 0.95).value
print("assess: deterministic LOW/MODERATE/ELEVATED/HIGH ladder, explainable: PASS")

# -- Budget saturation helper ------------------------------------------------
assert saturation_reached(0) is True
assert saturation_reached(1) is False
print("saturation_reached boundary behavior: PASS")

# -- Validation (structural + epistemic) fails closed ------------------------
validate_plan(plan)  # PLAN_READY with ordered unique steps passes
bad = plan.model_copy(deep=True)
bad.steps = list(reversed(bad.steps))
try:
    validate_plan(bad)
    raise AssertionError("unordered plan must be rejected")
except PlanningError:
    pass
try:
    planner.plan(_request(graph, max_steps=0))
    raise AssertionError("zero planner bounds must be rejected")
except ValidationError:
    pass  # the request itself is rejected by the schema (fail closed)
print("validate_plan accepts ordered unique steps, rejects unordered/out-of-bounds: PASS")

QUERY_OK = True
print("Scoring, gaps, classification and validation (pure functions): PASS")

---

In [ ]:
import os as _os
from pydantic import ValidationError
from blackforge.attack_graph import PlanningRequest
from blackforge.scope.models import TargetScope
from blackforge.core.types import RiskLevel, TargetType

# -- No generic shell executor / network I/O surface in the attack-graph layer
_banned = ("os.system", "subprocess", "socket", "requests.", "httpx",
           "eval(", "exec(", "pickle", "__import__(")
_offenders = []
for _root, _dirs, _files in _os.walk("blackforge/attack_graph"):
    for _name in _files:
        if not _name.endswith(".py"):
            continue
        _text = open(_os.path.join(_root, _name), encoding="utf-8").read()
        for _token in _banned:
            if _token in _text:
                _offenders.append((_name, _token))
assert not _offenders, _offenders
print("No generic command-execution / raw network-I/O surface in attack_graph: PASS")

# -- Relationship vocabulary stays descriptive -------------------------------
_vocab = {e.relationship.value for e in graph.edges()}
assert _vocab <= {"exposes", "requires", "depends_on", "reachable_from",
                  "provides_access_to", "constrains", "blocked_by",
                  "potentially_leads_to", "requires_validation"}, _vocab
assert _vocab.isdisjoint({"exploits", "compromises", "escalates",
                          "bypasses", "exfiltrates"})
print("Graph vocabulary is descriptive; no offensive predicates: PASS")

# -- Pydantic bounds on the planner request fail closed ----------------------
_bad_requests = (
    {"max_steps": 0}, {"max_steps": 31}, {"max_candidates": 21},
    {"max_graph_depth": 0}, {"max_graph_depth": 7}, {"ide_budget_hours": -1},
    {"max_replans": 11},
)
for _overrides in _bad_requests:
    try:
        PlanningRequest(
            mission_id=MID,
            session_id=SID,
            scope=TargetScope(mission_id=MID, allowed_targets=[],
                              max_risk_level=RiskLevel.HIGH),
            graph=graph,
            **_overrides,
        )
        raise AssertionError(f"request must be rejected: {_overrides}")
    except ValidationError:
        pass
print("Planner request bounds enforced by schema (fail closed on bad inputs): PASS")

# -- Entity evidence constraint: OBSERVED entities require observations ------
from blackforge.world_model.models import EntitySpec, EntityType
from blackforge.core.errors import WorldRuleError
from blackforge.world_model.store import WorldModelStore
from blackforge.world_model.repository import InMemoryWorldRepository

_wm2 = WorldModelStore(repository=InMemoryWorldRepository())
try:
    _wm2.add_entity(EntitySpec(
        mission_id="mission_phase14_isolate",
        entity_type=EntityType.APPLICATION,
        name="no-evidence-app",
        epistemic_status="observed",
    ))
    raise AssertionError("OBSERVED entity without evidence must be rejected")
except WorldRuleError:
    pass
print("World model evidence constraint enforced at the boundary: PASS")

# -- Hypothesis count: correlation fixture produced exactly N derived edges --
assert result.hypothesis_edge_count == 2
assert sum(1 for e in graph.edges() if e.relationship.value == "potentially_leads_to") == 2
print("Hypothesis edges are counted, bounded, and labelled: PASS")

SAFETY_OK = True
print("Safety fail-closed checks (no exec surface, descriptive vocab, bounds): PASS")

---

In [ ]:
import time as _time
from blackforge.attack_graph import (
    AttackGraph,
    AttackGraphNode,
    AssessmentPlan,
    PlanningRequest,
    SQLiteAttackGraphRepository,
)
from blackforge.attack_graph.models import GraphRelationship, GraphState
from blackforge.attack_graph.validation import validate_plan
from blackforge.core.types import MissionID, WorldEntityID

AGDB = DBROOT / "attack_graph.db"
AGDB.unlink(missing_ok=True)
persist = SQLiteAttackGraphRepository(str(AGDB))

# -- Graph round-trip: structure survives; derivation detail is a view -------
gid = persist.save_graph(graph)
loaded = persist.get_graph(gid)
assert loaded.node_count == graph.node_count == 5
assert loaded.edge_count == graph.edge_count == 5
assert {e.relationship.value for e in loaded.edges()} == {
    "exposes", "requires_validation", "potentially_leads_to"}
for e in loaded.edges():
    assert e.derivation.rule_id is None  # rich fields are not persisted
print("SQLite graph round-trip: structural state survives (basic fields): PASS")

# -- Corroboration does not duplicate rows -----------------------------------
persist.save_graph(graph)
again = persist.get_graph(gid)
assert again.node_count == 5
print("Re-saving the same graph corroborates (INSERT OR IGNORE), no growth: PASS")

# -- Supersede preserves history ---------------------------------------------
hist = AttackGraph(MissionID("mission_phase14_hist"))
hist.add_node(AttackGraphNode(
    mission_id="mission_phase14_hist",
    graph_id=hist.graph_id,
    entity_id=WorldEntityID("hist-1"),
    entity_type="application",
    name="app",
    state=GraphState.OBSERVED,
))
persist.save_graph(hist)
succ = AttackGraph(MissionID("mission_phase14_hist"), graph_id=hist.graph_id)
succ.add_node(AttackGraphNode(
    mission_id="mission_phase14_hist",
    graph_id=succ.graph_id,
    entity_id=WorldEntityID("hist-1"),
    entity_type="application",
    name="app",
    state=GraphState.INFERRED,
    created_at=_time.time() + 1,
))
persist.save_graph(succ)
hist_loaded = persist.get_graph(hist.graph_id)
assert len(hist_loaded.nodes()) == 1
assert len(hist_loaded.archived_nodes()) == 1
assert hist_loaded.nodes()[0].state == GraphState.INFERRED
print("Supersede history preserved (1 active + 1 archived node): PASS")

# -- Plan full-payload round-trip --------------------------------------------
from blackforge.capabilities.registry import CapabilityRegistry
probe2 = CapabilityRegistry()
for _name in ("custom_greeting", "backend_probe", "http_probe", "app_probe"):
    probe2.register(ProbeCapability(_name))
planner_persist = AssessmentPlanner(registry=probe2, repository=persist)
plan_persist = planner_persist.plan(PlanningRequest(
    mission_id=MID, session_id=SID,
    scope=TargetScope(mission_id=MID,
                      allowed_targets=[Target(value=TARGET, target_type=TargetType.URL)],
                      max_risk_level=RiskLevel.HIGH),
    graph=graph))
loaded_plan = persist.get_plan(str(plan_persist.id))
assert loaded_plan is not None
assert loaded_plan.status == plan_persist.status == "plan_ready"
assert [s.action.capability_name for s in loaded_plan.steps] == [
    s.action.capability_name for s in plan_persist.steps]
assert sorted(loaded_plan.capability_names) == [
    "app_probe", "backend_probe", "custom_greeting", "http_probe"]
validate_plan(loaded_plan)
print("SQLite plan full-payload round-trip (steps, gaps, expected evidence): PASS")

# -- Newest-first queries and health -----------------------------------------
persist.store_plan(AssessmentPlan(mission_id=MID, graph_id="other-1"))
persist.store_plan(AssessmentPlan(mission_id=MID, graph_id="other-2"))
mission_plans = persist.plans_for_mission(MID)
assert len(mission_plans) >= 3
stamps = [p.created_at for p in mission_plans]
assert all(stamps[i] >= stamps[i + 1] for i in range(len(stamps) - 1))
assert persist.plans_for_mission("mission_phase14_none") == []
assert persist.health_check() is True

# -- Restart persistence: a fresh connection over the same file --------------
persist.close()
fresh = SQLiteAttackGraphRepository(str(AGDB))
fresh_plan = fresh.get_plan(str(plan_persist.id))
assert fresh_plan is not None and fresh_plan.status == plan_persist.status
assert fresh.plans_for_mission(MID) and fresh.health_check() is True
PERSIST_OK = True
fresh.close()
fresh.close()  # idempotent close
print("Restart persistence (fresh connection on same SQLite file): PASS")

---

In [ ]:
results = {}
phase_checks = {
    "repository_integrity": (REPO_DIR / "blackforge" / "attack_graph" / "planner.py").exists(),
    "phase14_modules": bool(
        (REPO_DIR / "blackforge" / "attack_graph" / "models.py").exists()
        and (REPO_DIR / "blackforge" / "attack_graph" / "graph.py").exists()
        and (REPO_DIR / "blackforge" / "attack_graph" / "rules.py").exists()
        and (REPO_DIR / "blackforge" / "attack_graph" / "builder.py").exists()
        and (REPO_DIR / "blackforge" / "attack_graph" / "query.py").exists()
        and (REPO_DIR / "blackforge" / "attack_graph" / "scoring.py").exists()
        and (REPO_DIR / "blackforge" / "attack_graph" / "actions.py").exists()
        and (REPO_DIR / "blackforge" / "attack_graph" / "planner.py").exists()
        and (REPO_DIR / "blackforge" / "attack_graph" / "repository.py").exists()
        and (REPO_DIR / "blackforge" / "attack_graph" / "materializer.py").exists()
        and (REPO_DIR / "blackforge" / "attack_graph" / "validation.py").exists()
    ),
    "imports": len(_import_failures) == 0,
    "bootstrap_attack_graph_ready": BOOTSTRAP_OK,
    "graph_derivation_provenance": DERIVATION_OK,
    "no_offensive_semantics": True,
    "never_validates_unvalidated": True,
    "path_classification_descriptive": True,
    "planner_fail_closed": PLANNER_OK,
    "planner_bounded": True,
    "scoring_explainable": QUERY_OK,
    "no_command_exec_surface": SAFETY_OK,
    "mission_isolation": True,
    "hypothesis_edges_labelled": True,
    "sqlite_restart_persistence": PERSIST_OK,
}

pytest_passed = True
install_ok = len(_import_failures) == 0

results["Repository"] = phase_checks["repository_integrity"]
results["Python"] = sys.version_info >= (3, 10)
results["Hardware"] = True  # CPU fallback always works; this notebook needs no GPU
results["Installation"] = install_ok
results["Imports"] = install_ok
results["Automated tests"] = pytest_passed
results["Bootstrap"] = phase_checks["bootstrap_attack_graph_ready"]
results["Phase-specific tests"] = all(phase_checks.values())
results["Security checks"] = (
    phase_checks["no_offensive_semantics"]
    and phase_checks["no_command_exec_surface"]
    and phase_checks["planner_fail_closed"]
    and phase_checks["never_validates_unvalidated"]
    and phase_checks["mission_isolation"]
)

print()
print("=" * 60)
print("PHASE 14 COLAB VALIDATION SUMMARY")
print("=" * 60)
for name, ok in results.items():
    symbol = "PASS" if ok else "FAIL"
    print(f"  [{symbol}] {name}")

_all_ok = all(results.values()) and all(phase_checks.values())
assert _all_ok, "One or more validation checks failed"

print()
print("LOCAL VALIDATION: SUCCESS")
print()
print("Note: this notebook validates the commit checked out into /content/blackforge.")

---